# Monthly Product Health

Run at month end. Analyses the full calendar month and outputs an HTML report for
featured/demotion/cut decisions.

**Run All** — no manual config needed (auto-detects current month).

In [0]:
# ── Setup ─────────────────────────────────────────────────────────────────────

import sys
sys.path.append('/Workspace/Users/davidgao734@gmail.com/boba-cafe/POS')
sys.path.append('/Workspace/Users/davidgao734@gmail.com/boba-cafe/weekly-analysis')

import calendar
from datetime import datetime, date
from pipeline.transforms import _get_spark

spark = _get_spark()

# Auto-detect: analyse the most recently completed full month.
# Override MONTH and YEAR below to run for a specific month.
MONTH = None   # e.g. 3  (March).  None = auto
YEAR  = None   # e.g. 2026.        None = auto

today = datetime.now()
# Default to previous month so the report is always complete
if MONTH is None or YEAR is None:
    first_of_this = date(today.year, today.month, 1)
    last_month    = date(first_of_this.year, first_of_this.month, 1)
    import datetime as _dt
    last_month = first_of_this - _dt.timedelta(days=1)
    MONTH, YEAR = last_month.month, last_month.year

month_days   = calendar.monthrange(YEAR, MONTH)[1]
month_start  = date(YEAR, MONTH, 1).strftime('%Y-%m-%d')
month_end    = date(YEAR, MONTH, month_days).strftime('%Y-%m-%d')

prior_year   = YEAR if MONTH > 1 else YEAR - 1
prior_month  = MONTH - 1 if MONTH > 1 else 12
prior_days   = calendar.monthrange(prior_year, prior_month)[1]
prior_start  = date(prior_year, prior_month, 1).strftime('%Y-%m-%d')
prior_end    = date(prior_year, prior_month, prior_days).strftime('%Y-%m-%d')

month_label  = datetime(YEAR, MONTH, 1).strftime('%B %Y')
prior_label  = datetime(prior_year, prior_month, 1).strftime('%B %Y')

print(f'Current month : {month_start} → {month_end}  ({month_label})')
print(f'Prior month   : {prior_start} → {prior_end}  ({prior_label})')

In [0]:
# ── Decision thresholds (edit here) ──────────────────────────────────────────

MONTHLY_CFG = {
    # Promote: non-featured product must earn at least this much in the month
    'PROMOTE_MIN_REV':    5_000,
    # ... and at least this fraction of the subcat's featured-product average
    'PROMOTE_REL_SUBCAT': 0.50,

    # Demote: featured product earns <= this fraction of its subcategory average
    'DEMOTE_REL_SUBCAT':  0.50,
    # ... or its MoM growth is below this (−0.25 = −25 %)
    'DEMOTE_MOM_PCT':    -0.25,

    # Cut: non-featured product earns at most this much for the month
    'CUT_MAX_REV':        1_500,
    # ... and its MoM growth is below this (or it had no prior-month sales)
    'CUT_MOM_PCT':       -0.30,
}

In [0]:
# ── Load data ─────────────────────────────────────────────────────────────────

from config import TRANSACTIONS_TABLE, HIERARCHY_CSV, ANALYSIS_HTML_DIR
from modules.loader import load_transactions, load_product_hierarchy

cur_txn   = load_transactions(spark, TRANSACTIONS_TABLE, month_start, month_end)
pri_txn   = load_transactions(spark, TRANSACTIONS_TABLE, prior_start, prior_end)
hierarchy = load_product_hierarchy(HIERARCHY_CSV)

print(f'Transactions  this month  : {len(cur_txn):,} rows')
print(f'Transactions  prior month : {len(pri_txn):,} rows')
print(f'Product hierarchy loaded  : {"yes" if not hierarchy.empty else "no — raw product names only"}')

In [0]:
# ── Build report ──────────────────────────────────────────────────────────────

from modules import monthly_health

report_md = monthly_health.build(
    cur_txn, pri_txn, hierarchy,
    month_label, prior_label,
    MONTHLY_CFG,
)

print('Report built successfully')
for line in report_md.splitlines()[:40]:
    print(line)

In [0]:
# ── Save HTML ─────────────────────────────────────────────────────────────────

import os
from modules.utils import md_to_html

base_name    = f'{YEAR}-{MONTH:02d}_product_health'
report_title = f'Product Health: {month_label}'

os.makedirs(ANALYSIS_HTML_DIR, exist_ok=True)
html_path = os.path.join(ANALYSIS_HTML_DIR, f'{base_name}.html')
with open(html_path, 'w', encoding='utf-8') as f:
    f.write(md_to_html(report_md, title=report_title))

print(f'HTML → {html_path}')